# 1. Reading the Dataset.



In [29]:
import pandas as pd

In [30]:
# Data reading
def read_data(path):
    """To read a csv file and return a pandas df"""
    df = pd.read_csv(path)
    return df

In [31]:
from dotenv import load_dotenv
import os

load_dotenv()

df_true_csv_path = os.getenv('true_csv')
df_fake_csv_path = os.getenv('fake_csv')

In [32]:
df_true = read_data(df_true_csv_path)
df_fake = read_data(df_fake_csv_path)

In [33]:
# Explore the first rows of the real-news dataset
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [34]:
# Explore the first rows of the fake-news dataset
df_fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [35]:
# Add a "label" column to each DataFrame to identify the articles
df_true["label"] = "REAL"
df_fake["label"] = "FAKE"

In [36]:
# Unimos ambos DF en uno solo
df = pd.concat([df_true, df_fake], ignore_index=True)

In [37]:
# Verify the result
print(f'Real news: {len(df_true)}')
print(f'Fake news: {len(df_fake)}')
print(f'Total news articles: {len(df)}')

Real news: 21417
Fake news: 23481
Total news articles: 44898


In [38]:
df.head()

,title,text,subject,date,label
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017",REAL
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017",REAL
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017",REAL
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017",REAL
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017",REAL


In [39]:
df.tail()

,title,text,subject,date,label
44893,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016",FAKE
44894,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016",FAKE
44895,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016",FAKE
44896,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016",FAKE
44897,10 U.S. Navy Sailors Held by Iranian Military ...,21st Century Wire says As 21WIRE predicted in ...,Middle-east,"January 12, 2016",FAKE


# 2. Dataset Exploration.

In [40]:
# News distribution by label
df["label"].value_counts()

label
FAKE    23481
REAL    21417
Name: count, dtype: int64

In [41]:
# Example of a REAL article.
print("="* 80)
print("REAL ARTICLE")
print("="* 80)
print(f"\nTitle: {df[df['label']=='REAL'].iloc[0]['title']}")
print(f"\nText: {df[df['label']=='REAL'].iloc[0]['text'][:500]}...")

REAL ARTICLE

Title: As U.S. budget fight looms, Republicans flip their fiscal script

Text: WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way among Republicans, U.S. Representative Mark Meadows, speaking on CBS’ “Face the Nation,” drew a hard line on federal spending, which lawmakers are bracing to do battle over in January. When they retur...


In [42]:
# Example of a FAKE article.
print("="* 80)
print("FAKE ARTICLE")
print("="* 80)
print(f"\nTitle: {df[df['label']=='FAKE'].iloc[0]['title']}")
print(f"\nText: {df[df['label']=='FAKE'].iloc[0]['text'][:500]}...")

FAKE ARTICLE

Title:  Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing

Text: Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, a Happy and Healthy New Year,  President Angry Pants tweeted.  2018 will be a gr...


> **Observation.** Many fake-news articles contain residual HTML, URLs, special characters, and other artifacts that are not part of the article body. This noise is expected and should be removed.

# 3. Text Preprocessing.
Raw articles are not clean, and they may contain:
* **HTML markup** such as $<p>, <br>, <div>$, etc.
* URLs
* Punctuation that does not add useful signal.
* High-frequency words that appear across most articles and do not help separate fake from real news (for example: "the", "is", "and", "a").

## 3.1 HTML Removal.
Web-scraped articles often include residual HTML markup.

In [43]:
#  Warnings.
import warnings
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning # bs4 includes BeautifulSoup, which makes HTML parsing straightforward.

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

In [44]:
# Example text with HTML
ejemplo_html = '<o> Thi is a <b> breacking</b> new <a href="https://example.com">story</a><p>'
print("Original text: ",ejemplo_html)

Original text:  <o> Thi is a <b> breacking</b> new <a href="https://example.com">story</a><p>


In [45]:
# Remove the HTML tags
texto_limpio = BeautifulSoup(ejemplo_html,"html.parser").get_text()
print("Clean text:", texto_limpio)

Clean text:  Thi is a  breacking new story


In [46]:
# Helper function to remove HTML tags.
def strip_html(text):
  """Remove HTML tags from a text."""
  return BeautifulSoup(text, "html.parser").get_text()

In [47]:
# Test it on a real article from the dataset.
print(strip_html(df.iloc[0]["text"])[:300])

WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


## 3.2 URL Removal.
Some articles may also contain embedded URLs.

Regex pattern used for URL detection:
```
https?://\S+
```

In [48]:
import re

In [49]:
# Example text with a URL
ejemplo_url = "Breaking news abotu the economy https://www.example.com/news check it out"
print("Original text: ",ejemplo_url)

Original text:  Breaking news abotu the economy https://www.example.com/news check it out


In [50]:
# Remove URLs
texto_sin_urls = re.sub(r"https?://\S+", "", ejemplo_url)
print("Clean text:", texto_sin_urls)

Clean text: Breaking news abotu the economy  check it out


In [51]:
# Helper function
def remove_urls(text):
  """Remove URLs from a text."""
  return re.sub(r"https?://\S+", "", text)

## 3.3 Punctuation Removal and Lowercasing

In [52]:
# The string module provides common string utilities
import string

# Punctuation characters recognized by Python
print(string.punctuation)

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~


In [53]:
# Define the function
# def remove_punctuation(text):
#   """Remove punctuation from a text."""
#   return text.translate(str.maketrans("", "", string.punctuation))

def remove_punctuaction(text):
  """"Converts the text to lowercase and removes punctuation."""
  text = text.lower()
  # Remove standard ASCII punctuation
  text = text.translate(str.maketrans("", "", string.punctuation))
  # Also remove typographic quotes and other special Unicode characters
  # that are not included in string.punctuation
  text = re.sub(r"[\u2018\u2019\u201c\u201D\u2013\u2014\u2026]", "", text)
  return text

In [54]:
# Example
ejemplo = "Breaking News! THE president Said: 'No Coomment'"
print("Original text: ", ejemplo)
print("Clean text: ", remove_punctuaction(ejemplo))

Original text:  Breaking News! THE president Said: 'No Coomment'
Clean text:  breaking news the president said no coomment


## 3.4 Stopword Removal and Stemming.

Core preprocessing steps in **Natural Language Processing (NLP)**.

**Stopwords** are high-frequency words that appear in nearly every document and contribute little discriminative value. Common English examples include "the", "is", "and", "a", "in", and "to".

If the goal is to distinguish fake from real news, a word like "the" appears with similar frequency in both classes and adds little value. By contrast, terms such as "hoax" or "verified" can carry much stronger signal.

**Stemming** reduces a word to its **root** form. For example:
* "running", "runs", "ran" -> **"run"**

Why does this matter? For the model, "running" and "runs" are different tokens even though they share the same base meaning. Mapping them to a common stem groups morphological variants together.

> `nltk` library

* `PorterStemmer()` class

* stopword list via `nltk.corpus.stopwords.words('english')`

* Tokenization via `nltk.tokenize.word_tokenize()`

In [55]:
import nltk

# Download the required NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

ModuleNotFoundError: No module named 'nltk'

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

In [ ]:
stop_words = set(stopwords.words('english'))
print(f"Number of stopwords: {len(stop_words)}")
print(f"\nPrimeras 10 stopwords: {list(stop_words)[:20]}")

Número de stopwords: 198

Primeras 10 stopwords: ["you'd", "we'll", 'this', 'has', 'doesn', "doesn't", "isn't", 'our', 'don', 'both', 'such', 'them', 'wasn', 'yours', 're', "i'm", 'or', "she'll", 'being', 'few']


In [ ]:
# Proceso de Stemmer
stemmer = PorterStemmer()

palabras = ["running", "runs", "ran", "playing", "played", "investigation", "investigating"]
for palabra in palabras:
  print(f"{palabra:20s} -> {stemmer.stem(palabra)}")
#

running              -> run
runs                 -> run
ran                  -> ran
playing              -> play
played               -> play
investigation        -> investig
investigating        -> investig


In [ ]:
# Example completo: tokenizar, eliminar stopwords y aplicar stemming
ejemplo = "The president was running an investigation into the reported claims"
print("Original text: ", ejemplo)

# 1. Tokenize (split words)
tokens = word_tokenize(ejemplo.lower())
print("\nTokens: ", tokens)

# 2. Remove stopwords.
tokens_filtrados = [t for t in tokens if t not in stop_words]
print("\nTokens filtrados sin stopwords: ", tokens_filtrados)

# 3. Apply stemming
tokens_stemmed = [stemmer.stem(t) for t in tokens_filtrados]
print("\nTokens Stemmed: ", tokens_stemmed)


texto original:  The president was running an investigation into the reported claims

Tokens:  ['the', 'president', 'was', 'running', 'an', 'investigation', 'into', 'the', 'reported', 'claims']

Tokens filtrados sin stopwords:  ['president', 'running', 'investigation', 'reported', 'claims']

Tokens Stemmed:  ['presid', 'run', 'investig', 'report', 'claim']


## 4. Full Preprocessing Function.


In [ ]:
def preprocesar_texto(text):
  """
Apply all preprocessing transformations to a text:
0. Remove source prefixes
1. Remove HTML tags
2. Remove URLs
3. Lowercase and
4. Remove punctuation
5. Tokenize
6. Remove stopwords
7. Apply stemming
  """
  # 0. Remove source prefixes such as "City (Reuters) -" or "CITY (AP)"
  text = re.sub(r'^[A-Z\s,.]+\([^)]+\)\s*[-]?\s*', '',text)
  # 1. Remove HTML
  text = strip_html(text)
  # 2. Remove URLs
  text = remove_urls(text)
  # 3 and 4 lowercase and punctuation
  text = remove_punctuaction(text)
  # 5. Tokenize
  tokens = word_tokenize(text)
  # 6. Remove stopwords
  tokens = [t for t in tokens if t not in stop_words]
  # 7. Apply stemming
  stemmer = PorterStemmer()
  tokens = [stemmer.stem(t) for t in tokens]
  # Join tokens
  return " ".join(tokens)

In [ ]:
# Test with an article
texto_original = df.iloc[0]["text"]
print("Original text (first 300 characters):")
print(texto_original[:300])
print("\n"+ "="*80)
print("\nPreprocessed text (first 300 characters):")
print(preprocesar_texto(texto_original)[:300])

Texto Original (primeros 300 caracteres):
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


Texto Preprocesado (primeros 300 caracteres):
head conserv republican faction us congress vote month huge expans nation debt pay tax cut call fiscal conserv sunday urg budget restraint 2018 keep sharp pivot way among republican us repres mark meadow speak cb face nation drew hard line feder spend lawmak brace battl januari return holiday wednes


## 5. Apply Preprocessing to the Dataset

> Hint: use pandas `.apply()` to run the preprocessing function over the `text` column of the DataFrame. To start, work with a subset using `.sample()` or by selecting the first N rows.

In [ ]:
# Sample of 1,000 articles
# Shuffle the DataFrame so fake and real news are mixed.
df_sample = df.sample(n=1000, random_state=42)

print(f'Sample size: {len(df_sample)}')
print(f'Label distribution')
print(df_sample["label"].value_counts())

tamaño del subconjunto: 1000
Distribución de Etiquetas
label
FAKE    536
REAL    464
Name: count, dtype: int64


In [ ]:
# Apply preprocessing to all articles in the sample
print("Preprocessing articles...")
df_sample["text_clean"] = df["text"].apply(preprocesar_texto)
print("Done")

Preprocesando noticias...
Listo


In [ ]:
# Result
df_sample[["text","text_clean", "label"]].head()

,text,text_clean,label
22216,"Donald Trump s White House is in chaos, and th...",donald trump white hous chao tri cover russia ...,FAKE
27917,Now that Donald Trump is the presumptive GOP n...,donald trump presumpt gop nomine time rememb c...,FAKE
25007,Mike Pence is a huge homophobe. He supports ex...,mike penc huge homophob support exgay convers ...,FAKE
1377,SAN FRANCISCO (Reuters) - California Attorney ...,california attorney gener xavier becerra said ...,REAL
32476,Twisted reasoning is all that comes from Pelos...,twist reason come pelosi day especi 2006 promi...,FAKE


In [ ]:
# Before and after example
print("Before")
print(df_sample.iloc[0]["text"][:200])
print("After")
print(df_sample.iloc[0]["text_clean"][:200])


Antes
Donald Trump s White House is in chaos, and they are trying to cover it up. Their Russia problems are mounting by the hour, and they refuse to acknowledge that there are problems surrounding all of th
Después
donald trump white hous chao tri cover russia problem mount hour refus acknowledg problem surround fake news hoax howev fact bear thing differ seem crack congression public leadershipchuck grassley ri


This is one of the most important steps and one of the most conceptually new ones. Up to this point, the datasets used in the course had numeric columns (price, number of teams, etc.). Here, however, the input is **text**, and ML models require **numbers**.

How do we convert text into numbers? This is called **vectorization**, and there are several approaches. Here we use one of the simplest and most interpretable: **Bag of Words**.

#### What is Bag of Words?

The idea is simple. Imagine we only have two articles in our dataset:

- **Article 1**: *"president signs new law today"*
- **Article 2**: *"new study shows results today"*

Bag of Words works as follows:

1. **Build a vocabulary** with all unique words in the dataset: `["law", "new", "president", "results", "shows", "signs", "study", "today"]` -> input features

2. **Represent each article as a numeric vector**, where each value indicates **how many times each vocabulary term appears** in that article:

| | law | new | president | results | shows | signs | study | today |
|---|---|---|---|---|---|---|---|---|
| **Article 1** | 1 | 1 | 1 | 0 | 0 | 1 | 0 | 1 |
| **Article 2** | 0 | 1 | 0 | 1 | 1 | 0 | 1 | 1 |

Each article becomes a **numeric vector** that the ML model can understand. This vectorization step is fundamental for text workflows.

We will go deeper into these techniques later in the course. For now, we will use Sklearn's implementation.

<div style="background-color:#D9EEFF;color:black;padding:2%;">
Apply Bag of Words vectorization to the processed text so it can be transformed into a numerical representation consumable by the ML model.
</div>

**Hint**: Inspect Sklearn's `CountVectorizer`. It implements the Bag of Words workflow described above and exposes two main methods:
- `.fit()`: learns the vocabulary from the training texts.
- `.transform()`: converts texts into numeric vectors using the learned vocabulary.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer # cuenta vectore xd

In [ ]:
# Example para entender como CountVectorizer.
ejemplo_textos = [
    "president signs new law today",
    "new study shows results today"
]

vectorizer_ejemplo = CountVectorizer()
vectorizer_ejemplo.fit(ejemplo_textos)

print("Learned vocabulary:")
print(vectorizer_ejemplo.get_feature_names_out()) # Accessing the learned feature names


Vocabulario aprendido:
['law' 'new' 'president' 'results' 'shows' 'signs' 'study' 'today']


In [ ]:
# Text-to-vector transformation.
vectores_ejemplo = vectorizer_ejemplo.transform(ejemplo_textos)

# Visualize the resulting matrix

pd.DataFrame(
    vectores_ejemplo.toarray(), # Convert to a pandas DataFrame so the matrix is easier to inspect than the raw array
    columns=vectorizer_ejemplo.get_feature_names_out(),
    index=["Article 1", "Article 2"]


)

,law,new,president,results,shows,signs,study,today
Noticia 1,1,1,1,0,0,1,0,1
Noticia 2,0,1,0,1,1,0,1,1


As shown above, the result matches the table described earlier. Each row is an article and each column is a vocabulary term. The values indicate how often each term appears in each article.

Now we will apply the same process to the real dataset.

In [ ]:
# Apply CountVectorizer to the processed texts
vectorizer = CountVectorizer()
vectorizer.fit(df_sample["text_clean"]) # Learn the vocabulary

print(f'Vocabulary size: {len(vectorizer.get_feature_names_out())}') # Input feature count

Tamaño del vocabulario: 19184


In [ ]:
# Text-to-vector transformation
X_vect = vectorizer.transform(df_sample["text_clean"])

print(f"Resulting matrix shape: {X_vect.shape}")
print(f"  → {X_vect.shape[0]} articles")
print(f"  → {X_vect.shape[1]} vocabulary terms")

Dimensiones de la matriz resultante: (1000, 19184)
  → 1000 noticias
  → 19184 palabras en el vocabulario


In [ ]:
# Inspect the first rows of the matrix (only the first 10 columns)
pd.DataFrame(
    X_vect.toarray(),
    columns=vectorizer.get_feature_names_out()
)

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Note**: The resulting matrix is large (each article is represented by a vector with thousands of elements) and mostly sparse (a single article contains only a small fraction of the vocabulary). Sklearn handles sparse matrices efficiently internally.

## 7. Model Training.


Now we train a **Logistic Regression** model to classify articles as fake or real. The workflow mirrors the linear-regression example shown earlier: we provide labeled training data and the algorithm learns the patterns that separate the two classes.


<div style="background-color:#D9EEFF;color:black;padding:2%;">
Use the <code>LogisticRegression</code> ML algorithm to classify fake vs. real articles. Its implementation is available in Sklearn.
</div>



**Hint 1**: Start with a smaller subset so the notebook runs quickly, for example 1,000 articles. Apply all the preprocessing steps defined above.

In [ ]:
# Random sample of 1,500 articles
df_all = df.sample(n=1500, random_state=42)

# Select 1,000 articles for training
df_sample = df_all.iloc[:1000]

print(f"Sample size: {len(df_sample)}")
print(f"\nLabel distribution:")
print(df_sample["label"].value_counts())

Tamaño del subconjunto: 1000

Distribución de etiquetas:
label
FAKE    536
REAL    464
Name: count, dtype: int64


In [ ]:
# Apply preprocessing
print("Preprocessing articles...")
df_sample["text_clean"] = df_sample["text"].apply(preprocesar_texto)
print("Done!")

Preprocesando noticias...
¡Listo!


/tmp/ipykernel_3667/1375928415.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample["text_clean"] = df_sample["text"].apply(preprocesar_texto)


**Hint 2:** Vectorize the dataset to obtain a numerical representation of the articles.

In [ ]:
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(df_sample["text_clean"])


In [ ]:
print(X_train.toarray())
print("\nFeatures:", len(vectorizer.get_feature_names_out()))

[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]

Features: 19184


In [ ]:
pd.DataFrame(X_train.toarray(), columns=[vectorizer.get_feature_names_out()])

,000004,00009,0006,005380k,0100,015760k,020,0259,0300,0307,...,zip,zipper,zippi,zolani,zombi,zone,zoran,zuckerberg,zuma,zweli
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
y_train = df_sample["label"]
y_train

,label
22216,FAKE
27917,FAKE
25007,FAKE
1377,REAL
32476,FAKE
...,...
27000,FAKE
25165,FAKE
20813,REAL
11756,REAL


**Hint 3**: Train Sklearn's `LogisticRegression` model. This is a supervised learning algorithm.

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(max_iter=1000) # max_iter controls the optimization loop (gradient descent)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## 8. Prediction

**Hint**: Review the `predict()` method exposed by `LogisticRegression`. When scoring a new article, apply the same preprocessing steps used during training. **When vectorizing, use only the `transform()` method from `CountVectorizer`**.

##### Loading a new set of articles

In [ ]:
# Take the 500 articles that were not used to train the model
df_test = df_all.iloc[1000:]

print("Preprocessing test articles...")
df_test["text_clean"] = df_test["text"].apply(preprocesar_texto) # Predict whether the new article is real or fake; preprocessing is the same as above
print("Done!")


Preprocesando noticias de test...
¡Listo!


/tmp/ipykernel_3667/1614974129.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_test["text_clean"] = df_test["text"].apply(preprocesar_texto) # Predicimos si la noticia nueva es verdadera o falsa. Por ende hacemos el proceso de preprocesamiento de texto


In [ ]:
X_test = df_test["text_clean"]
y_test = df_test["label"]

print(f"Test articles: {len(X_test)}")

Noticias de test: 500


In [ ]:
X_test

,text_clean
2851,us senat 2 republican said monday hope vote co...
1000,presid bashar alassad famili role futur syria ...
44165,stori show conscienti journalist new york time...
26668,mark cuban much richer donald trump also sane ...
20162,reuter florida power light said wednesday prov...
...,...
37464,wonder violenc like today shoot happen left st...
28616,mani us elect presid obama offic amaz thing pa...
31863,one month elect polit activist hillari support...
14495,iraqi forc friday captur border town rawa last...


##### Preprocess the articles with the vectorizer created earlier

In [ ]:
# Apply CountVectorizer (only .transform(), NO .fit())
X_test = vectorizer.transform(X_test) # The vectorizer already contains the learned vocabulary; calling fit here would overwrite it.

##### Predicting the article class

In [ ]:
y_pred = clf.predict(X_test)
y_pred

array(['REAL', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL',
       'REAL', 'REAL', 'REAL', 'FAKE', 'REAL', 'REAL', 'REAL', 'FAKE',
       'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'REAL', 'REAL', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'FAKE',
       'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL',
       'FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'REAL', 'FAKE', 'REAL',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'REAL', 'FAKE', 'REAL', 'REAL',
       'FAKE', 'FAKE', 'REAL', 'REAL', 'REAL', 'FAKE', 'FAKE', 'FAKE',
       'FAKE', 'REAL', 'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'FAKE',
       'FAKE', 'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL', 'REAL', 'FAKE',
       'FAKE', 'FAKE', 'REAL', 'FAKE', 'REAL', 'FAKE', 'FAKE', 'REAL',
      

In [ ]:
print("Prediction:\n", y_pred)
print("\nTrue labels:\n", y_test.values)

Predicción:
 ['REAL' 'REAL' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE'
 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'REAL' 'FAKE'
 'REAL' 'REAL' 'REAL' 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE'
 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'REAL' 'FAKE'
 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE'
 'REAL' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL'
 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL' 'REAL'
 'FAKE' 'FAKE' 'REAL' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL'
 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL'
 'FAKE' 'REAL' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE'
 'FAKE' 'REAL' 'FAKE' 'REAL' 'REAL' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'REAL'
 'REAL' 'FAKE' 'REAL' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'FAKE'
 'FAKE' 'FAKE' 'FAKE' 'FAKE' 'REAL' 'FAKE' 'REAL' 'REAL' 'REAL' 

##### Evaluating the results

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred)))

Accuracy: 0.950


## 9. Scaling Up the Dataset.

> ML algorithms perform better when they have more training data.

In [ ]:
# Sample of 42,000 articles
df_grande = df.sample(n=42000, random_state=42)

print("Preprocessing articles...")
df_grande["text_clean"] = df_grande["text"].apply(preprocesar_texto)
print("Done!")

Preprocesando noticias...
¡Listo!


In [ ]:
# Use 40,000 articles to train the model and 2,000 for testing.
X_train, y_train = df_grande["text_clean"][:40000], df_grande["label"][:40000]
X_test, y_test = df_grande["text_clean"][40000:], df_grande["label"][40000:]

print(f'Training articles: {len(X_train)}')
print(f'Test articles: {len(X_test)}')

Noticias de entrenamiento: 40000
Noticias de test: 2000


In [ ]:
# Vectorize. Convert to numeric values.
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train) # Build the vocabulary and convert to numbers.

In [ ]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 6169000 stored elements and shape (40000, 182468)>

In [ ]:
# Entrenando
clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
X_test = vectorizer.transform(X_test) # Use transform only because the vocabulary was learned from X_train.

In [ ]:
y_pred = clf.predict(X_test)

In [ ]:
print("Accuracy: {:.3f}".format(accuracy_score(y_test, y_pred)))

Accuracy: 0.987


## 10. Testing with New Articles

**Hint**: When classifying a new article, you must:
1. Apply the `preprocesar_texto()` function to it.
2. Apply **only** the `.transform()` method from `CountVectorizer` (NOT `.fit()`).
3. Use the trained model's `.predict()` method.
4. Optionally, use `.predict_proba()` to inspect the probabilities assigned to each class.

In [ ]:
# Define a few test articles
noticias_nuevas = [
    """DUBAI, Feb 13 (Reuters) - Dubai port giant DP World said on Friday its chairman and chief executive Sultan Ahmed Bin Sulayem had resigned, an announcement that followed mounting pressure over his alleged ties to Jeffrey Epstein.
Bin Sulayem, one of the Middle East's most prominent business figures, is among the highest-profile executives to face scrutiny and be removed from senior roles following the recent release of the Epstein files.""",
    "EXPOSED: Secret documents reveal that the moon landing was filmed in a Hollywood basement by the CIA!!!",
    """WASHINGTON, Feb 12 (Reuters) - An Arizona sheriff is blocking FBI access to key evidence in the investigation into the abduction of U.S. television journalist Savannah Guthrie's mother, impairing its ability to assist in the probe, a U.S. law enforcement official with knowledge of the case told Reuters on Thursday.
The FBI asked Pima County Sheriff Chris Nanos for physical evidence in the case, including a glove and DNA from the home of 84-year-old Nancy Guthrie, to be processed at the FBI's national crime laboratory in Quantico, Virginia, but Nanos has insisted instead on using a private lab in Florida, the official said.""",
    "YOU WONT BELIEVE THIS: Politicians are secretly lizard people controlling the world government!!!"
]

In [ ]:
noticias_procesadas = [preprocesar_texto(n) for n in noticias_nuevas]

# Vectorize using the same trained vectorizer (only .transform())
noticias_vect = vectorizer.transform(noticias_procesadas)

# Make predictions
predicciones = clf.predict(noticias_vect)
probabilidades = clf.predict_proba(noticias_vect)

# Display the results
for noticia, pred, prob in zip(noticias_nuevas, predicciones, probabilidades):
    print(f"Article: {noticia[:80]}...")
    print(f"  → Prediction: {pred}")
    print(f"  → Fake probability: {prob[0]:.2%} | Real probability: {prob[1]:.2%}")
    print()

Noticia: DUBAI, Feb 13 (Reuters) - Dubai port giant DP World said on Friday its chairman ...
  → Predicción: REAL
  → Probabilidad FAKE: 5.34% | Probabilidad REAL: 94.66%

Noticia: EXPOSED: Secret documents reveal that the moon landing was filmed in a Hollywood...
  → Predicción: FAKE
  → Probabilidad FAKE: 99.46% | Probabilidad REAL: 0.54%

Noticia: WASHINGTON, Feb 12 (Reuters) - An Arizona sheriff is blocking FBI access to key ...
  → Predicción: REAL
  → Probabilidad FAKE: 0.12% | Probabilidad REAL: 99.88%

Noticia: YOU WONT BELIEVE THIS: Politicians are secretly lizard people controlling the wo...
  → Predicción: FAKE
  → Probabilidad FAKE: 92.43% | Probabilidad REAL: 7.57%

